# V4-5000 Evaluation (`gen_rp=1.00_fb1.08`)
Evaluate all 100 frozen validation cases on `finetuned_v4_5000`.

In [1]:
%pip install -q "transformers==4.57.6" "datasets==4.8.5" "peft==0.18.1" "accelerate==1.15.0" pandas huggingface_hub


In [2]:
import json, os, random, shutil, time
from pathlib import Path
import pandas as pd
import torch
from datasets import load_from_disk
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

os.chdir("/content/clinical_v4_5000")
import sys
sys.path.insert(0, str(Path.cwd()))
from scripts.build_v4_notebooks import digest, find_schema_violations, score_run_records

class GeneratedOnlyRepetitionPenalty(LogitsProcessor):
    def __init__(self, penalty: float, prompt_length: int):
        self.penalty = float(penalty)
        self.prompt_length = int(prompt_length)
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.penalty == 1.0 or input_ids.shape[1] <= self.prompt_length:
            return scores
        gen_ids = input_ids[:, self.prompt_length :]
        if gen_ids.shape[0] == 1:
            gen_ids = torch.unique(gen_ids[0]).unsqueeze(0)
        score = torch.gather(scores, 1, gen_ids)
        score = torch.where(score < 0, score * self.penalty, score / self.penalty)
        return scores.scatter(1, gen_ids, score)

root = Path.cwd()
contract = json.loads((root / "results_v4_5000/baseline/comparison_contract.json").read_text(encoding="utf-8"))
pdir = root / "prepared_clinical_dataset" / contract["dataset_version"]
cases = load_from_disk(str(pdir / "dataset"))["validation"].to_list()
assert [c["case_id"] for c in cases] == contract["case_ids"]
assert digest([c["prompt_text"] for c in cases]) == contract["prompt_sha256"]

out_dir = root / "results_v4_5000/comparison/finetuned_v4_5000__gen_rp_1.00"
out_dir.mkdir(parents=True, exist_ok=True)
gen_path, diag_path, raw_path = out_dir / "generations.json", out_dir / "generation_diagnostics.json", out_dir / "raw_rp1_generations.json"
generations = json.loads(gen_path.read_text(encoding="utf-8")) if gen_path.exists() else {}
diagnostics = json.loads(diag_path.read_text(encoding="utf-8")) if diag_path.exists() else {}
raw_gens = json.loads(raw_path.read_text(encoding="utf-8")) if raw_path.exists() else {}

token = os.environ.get("HF_TOKEN")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(contract["model"], revision=contract["base_model_revision"], token=token)
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
random.seed(contract["seed"])
torch.manual_seed(contract["seed"])
base_m = AutoModelForCausalLM.from_pretrained(contract["model"], revision=contract["base_model_revision"], token=token, dtype=torch.float32).to(device)
model = PeftModel.from_pretrained(base_m, str(root / "results_v4_5000/finetuned_v4_5000/adapter"), is_trainable=False).eval()
eos = model.generation_config.eos_token_id
eos_ids = eos if isinstance(eos, list) else [eos]
max_new = int(contract["max_new_tokens"])
started = time.monotonic()

for case in cases:
    key = case["case_id"]
    if generations.get(key, "").strip() and key in diagnostics:
        continue
    inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
    n_prompt = inputs.input_ids.shape[1]
    with torch.inference_mode():
        logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
        assert bool(torch.isfinite(logits).all().item())
        del logits
        out_rp1 = model.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=pad_id)
        ids_rp1 = out_rp1[0, n_prompt:].tolist()
        text_rp1 = tokenizer.decode(ids_rp1, skip_special_tokens=True)
        raw_gens[key] = text_rp1
        raw_path.write_text(json.dumps(raw_gens, ensure_ascii=False, indent=2), encoding="utf-8")
        hit_budget = len(ids_rp1) >= max_new
        fb_triggered = False
        final_ids, final_text = ids_rp1, text_rp1
        if hit_budget:
            fb_triggered = True
            out_fb = model.generate(
                **inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=pad_id,
                logits_processor=LogitsProcessorList([GeneratedOnlyRepetitionPenalty(1.08, n_prompt)]),
            )
            final_ids = out_fb[0, n_prompt:].tolist()
            final_text = tokenizer.decode(final_ids, skip_special_tokens=True)
    diagnostics[key] = {
        "prompt_tokens": n_prompt, "rp1_generated_tokens": len(ids_rp1), "rp1_reached_budget": hit_budget,
        "fallback_to_gen_rp_1.08_triggered": fb_triggered, "generated_tokens": len(final_ids),
        "finite_first_token_logits": True, "empty_output": not final_text.strip(),
        "reached_budget": len(final_ids) >= max_new, "ended_with_eos": bool(final_ids) and final_ids[-1] in eos_ids,
    }
    diag_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
    generations[key] = final_text
    gen_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[finetuned_v4_5000 | gen_rp_1.00] {len(generations)}/100 {key}: rp1_toks={len(ids_rp1)}, fb={fb_triggered}, final={len(final_ids)} ({time.monotonic()-started:.1f}s)", flush=True)

df_fb, sum_fb = score_run_records(cases, generations, diagnostics)
sum_fb["fallback_triggered_cases"] = sum(1 for c in cases if diagnostics[c["case_id"]].get("fallback_to_gen_rp_1.08_triggered"))
df_fb.to_csv(out_dir / "per_case.csv", index=False)
(out_dir / "aggregate.json").write_text(json.dumps(sum_fb, indent=2), encoding="utf-8")
(out_dir / "schema_violations.json").write_text(json.dumps(find_schema_violations(cases, generations), indent=2), encoding="utf-8")

base_sum = json.loads((root / "results_v4_5000/baseline/gen_rp_1.08/aggregate.json").read_text(encoding="utf-8"))
summary_rows = [{"model": "base", "decoding": "gen_rp_1.08", **base_sum}, {"model": "finetuned_v4_5000", "decoding": "gen_rp_1.00_fb1.08", **sum_fb}]
pd.DataFrame(summary_rows).to_csv(root / "results_v4_5000/comparison/summary_comparison.csv", index=False)
(root / "results_v4_5000/comparison/summary_comparison.json").write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")
archive = shutil.make_archive(str((root / "results_v4_5000/comparison").resolve()), "zip", root_dir=root / "results_v4_5000", base_dir="comparison")
print("V4-5000 COMPARISON COMPLETE:", json.dumps(sum_fb, indent=2), "archive=", archive, flush=True)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[finetuned_v4_5000 | gen_rp_1.00] 1/100 validation:2791: rp1_toks=441, fb=False, final=441 (25.2s)


[finetuned_v4_5000 | gen_rp_1.00] 2/100 validation:474: rp1_toks=420, fb=False, final=420 (47.2s)


[finetuned_v4_5000 | gen_rp_1.00] 3/100 validation:106: rp1_toks=529, fb=False, final=529 (74.4s)


[finetuned_v4_5000 | gen_rp_1.00] 4/100 validation:1180: rp1_toks=435, fb=False, final=435 (97.1s)


... (93 outputs left out)

[finetuned_v4_5000 | gen_rp_1.00] 98/100 validation:1146: rp1_toks=360, fb=False, final=360 (3091.3s)


[finetuned_v4_5000 | gen_rp_1.00] 99/100 validation:281: rp1_toks=455, fb=False, final=455 (3116.0s)


[finetuned_v4_5000 | gen_rp_1.00] 100/100 validation:895: rp1_toks=502, fb=False, final=502 (3142.9s)


V4-5000 COMPARISON COMPLETE: {
  "selected_cases": 100,
  "evaluated_cases": 100,
  "empty_cases": 0,
  "raw_json_object_valid_cases": 98,
  "json_object_valid_after_fence_removal_cases": 98,
  "exact_top_level_keys_cases": 98,
  "allowed_schema_valid_cases": 97,
  "raw_allowed_schema_valid_cases": 97,
  "exact_key_match_cases": 13,
  "placeholder_only_cases": 0,
  "reached_budget_cases": 1,
  "mean_value_precision": 0.39787035515251473,
  "mean_value_recall": 0.41568725270026446,
  "mean_value_f1": 0.40167404385323097,
  "mean_soft_value_precision": 0.546279173935204,
  "mean_soft_value_recall": 0.5738095786790985,
  "mean_soft_value_f1": 0.5527139761482683,
  "matched_values": 1636,
  "reference_values": 3856,
  "generated_values": 4044,
  "mean_generated_tokens": 500.16,
  "max_5gram_repeat_ge_3_cases": 43,
  "fallback_triggered_cases": 5
} archive= /content/clinical_v4_5000/results_v4_5000/comparison.zip
